# Classification Track — Insurance Claims Fraud Detection
**23CSE301 Machine Learning Capstone — Review 1 (Part A) + Review 2 (Part B)**

**Dataset:** `data/insurance_claims/insurance_claims.csv` (1,000 rows x 40 columns) — auto-insurance policy and incident records.

**Problem statement:** Predict **`fraud_reported`** (Yes/No) — whether an insurance claim is fraudulent — from policy, customer and incident attributes.

**Scope of this notebook:**

*Review 1 — Part A*
1. Data loading & audit
2. Exploratory Data Analysis (EDA)
3. Data cleaning, feature engineering, encoding, scaling, splitting
4. **Dimensionality reduction (PCA)** — fit on the training split, then used as the shared preprocessing for every Part-A algorithm
5. Training & evaluating the **Part-A algorithms**: Logistic Regression, K-Nearest Neighbors, Naive Bayes (Gaussian), Decision Tree, SVC
6. Preliminary comparison table (Accuracy, weighted F1, confusion matrix)

*Review 2 — Part B* (Sections E–I, after the Part-A demo)

7. Training the **Part-B algorithms** on the *same* split and pipeline: Random Forest, AdaBoost, Gradient Boosting, Bagging (Decision Tree base), MLP Classifier
8. **Consolidated 10-algorithm comparison table** (Accuracy, Precision, Recall, F1, ROC-AUC) + ROC curves for all ten
9. Diagnostic: **PCA vs. original encoded features** — why tree ensembles need the raw one-hot features
10. **Hyperparameter tuning** (`GridSearchCV`) for all 10 algorithms, with the feature representation itself treated as a hyperparameter
11. **Final model selection**, 5-fold cross-validation of the two best models, feature importance, and saved models

**Random seed** `42` is used everywhere for reproducibility.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 100

DATA_PATH = '../data/insurance_claims/insurance_claims.csv'

## Section A — Dataset Loading & Audit (A1)

In [2]:
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print()
print('Data types:')
print(df.dtypes.value_counts())

Shape: 1000 rows x 40 columns

Data types:
object     21
int64      17
float64     2
Name: count, dtype: int64


In [3]:
# The raw CSV encodes missing categorical values as the literal string '?'; convert to NaN
# so pandas/sklearn treat them as proper missing values.
df = df.replace('?', np.nan)

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values:')
print(missing)

print(f'\nDuplicate rows: {df.duplicated().sum()}')

print('\nTarget (fraud_reported) class distribution:')
print(df['fraud_reported'].value_counts())
print(df['fraud_reported'].value_counts(normalize=True).round(3))

Columns with missing values:
_c39                       1000
property_damage             360
police_report_available     343
collision_type              178
authorities_contacted        91
dtype: int64

Duplicate rows: 0

Target (fraud_reported) class distribution:
fraud_reported
N    753
Y    247
Name: count, dtype: int64
fraud_reported
N    0.753
Y    0.247
Name: proportion, dtype: float64
